"""
feature engineering for BodyFat.csv (extended dataset)

adds a 'NavyBodyFat' column using the US Navy circumference method,
and drops the 'Original' y/n column.

US Navy formula (imperial units: inches):
    Men:   %BF = 86.010 * log10(Abdomen - Neck) - 70.041 * log10(Height) + 36.76
    Women: %BF = 163.205 * log10(Waist + Hip - Neck) - 97.684 * log10(Height) - 78.387

"""

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("BodyFat.csv")

df = df.drop(columns="Original")
df["Height"] = df["Height"] *100
df["Height_in"] = df["Height"] / 2.54
df["Abdomen_in"] = df["Abdomen"] / 2.54
df["Neck_in"] = df["Neck"] / 2.54
df["Hip_in"] = df["Hip"] / 2.54 


def navy_bodyfat_male(abdomen, neck, height):
    return 86.010 * np.log10(abdomen - neck) - 70.041 * np.log10(height) + 36.76


def navy_bodyfat_female(waist, hip, neck, height):
    return 163.205 * np.log10(waist + hip - neck) - 97.684 * np.log10(height) - 78.387


def compute_navy(row):
    is_male = str(row["Sex"]).strip().upper()=="M"
    if is_male:
        return navy_bodyfat_male(row["Abdomen_in"], row["Neck_in"], row["Height_in"])
    else:
        return navy_bodyfat_female(row["Abdomen_in"], row["Hip_in"], row["Neck_in"], row["Height_in"])



df["NavyBodyFat"] = df.apply(compute_navy, axis=1)
df=df.drop(columns=["Hip_in","Neck_in","Abdomen_in","Height_in"])

df.to_csv("BodyFat_engineered.csv", index=False)